In [ ]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/asaniczka/1-3m-linkedin-jobs-and-skills-2024/job_skills.csv').dropna()

In [ ]:
import re

def dedup_text(text):
    phrases = [p.strip() for p in text.split(',') if p.strip()]
    return ', '.join(dict.fromkeys(phrases))


def cut_noise(text, max_words_per_phrase=5):
    # normalize dấu
    text = text.replace('.', ',').replace(';', ',')
    text = re.sub(r',+', ',', text) 
    
    phrases = [p.strip() for p in text.split(',') if p.strip()]
    
    phrases = [p for p in phrases if len(p.split()) <= max_words_per_phrase]
    
    return ', '.join(phrases)

df['job_skills'] = df['job_skills'].apply(dedup_text)
df['job_skills'] = df['job_skills'].apply(cut_noise)

In [ ]:
cols_keep = ['job_link', 'job_skills', 'job_title', 'job_level', 'job_type']
df2 = pd.read_csv('/kaggle/input/datasets/asaniczka/1-3m-linkedin-jobs-and-skills-2024/linkedin_job_postings.csv')
df3 = pd.merge(df, df2, on="job_link")[cols_keep]
df3 = df3.dropna()
df3

In [25]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import spacy
from spacy.cli import download

try:
    nlp = spacy.load("en_core_web_lg")
except OSError:
    print("Model en_core_web_lg chưa có, đang tải...")
    download("en_core_web_lg")
    nlp = spacy.load("en_core_web_lg")

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

lemmatizer = WordNetLemmatizer()
STOP_WORDS = set(stopwords.words('english'))

SYNONYM_MAP = {
    # ===== LEVEL =====
    "sr": "senior",
    "jr": "junior",
    "mid": "mid",
    "lead": "senior",
    "principal": "senior",
    "staff": "senior",
    "intern": "intern",
    "trainee": "junior",

    # ===== ROLE COMMON =====
    "mgr": "manager",
    "mngr": "manager",
    "dir": "director",
    "vp": "vice president",
    "svp": "senior vice president",
    "assoc": "associate",
    "asst": "assistant",
    "rep": "representative",
    "exec": "executive",
    "officer": "officer",
    "admin": "administrator",

    # ===== TECH =====
    "dev": "developer",
    "eng": "engineer",
    "swe": "software engineer",
    "frontend": "front end",
    "backend": "back end",
    "fullstack": "full stack",
    "full-stack": "full stack",
    "qa": "quality assurance",
    "sdet": "software development engineer in test",
    "ml": "machine learning",
    "ai": "artificial intelligence",
    "ds": "data scientist",
    "de": "data engineer",
    "fe": "front end",
    "be": "back end",

    # ===== DATA =====
    "analyst": "analyst",
    "analytics": "analyst",
    "bi": "business intelligence",
    "data analyst": "data analyst",
    "data analytics": "data analyst",

    # ===== BUSINESS =====
    "biz": "business",
    "bd": "business development",
    "bdr": "business development representative",
    "ba": "business analyst",
    "pm": "project manager",
    "pdm": "product manager",

    # ===== SALES =====
    "ae": "account executive",
    "am": "account manager",
    "sales rep": "sales representative",
    "salesperson": "sales representative",

    # ===== MARKETING =====
    "mkt": "marketing",
    "seo": "search engine optimization",
    "sem": "search engine marketing",
    "smm": "social media marketing",

    # ===== FINANCE =====
    "acct": "accountant",
    "acc": "accountant",
    "cpa": "certified public accountant",
    "fp&a": "financial planning analysis",

    # ===== HR =====
    "hr": "human resources",
    "hrbp": "human resources business partner",
    "recruiter": "recruiter",
    "talent": "talent acquisition",

    # ===== HEALTHCARE =====
    "rn": "registered nurse",
    "lpn": "licensed practical nurse",
    "np": "nurse practitioner",
    "pa": "physician assistant",
    "md": "medical doctor",
    "dr": "doctor",

    # ===== ENGINEERING (NON-IT) =====
    "mech": "mechanical",
    "elec": "electrical",
    "civil": "civil",
    "arch": "architect",
    "tech": "technician",

    # ===== OPERATIONS =====
    "ops": "operations",
    "coo": "chief operating officer",
    "logistics": "logistics",
    "supply chain": "supply chain",

    # ===== EDUCATION =====
    "prof": "professor",
    "lect": "lecturer",
    "ta": "teaching assistant",

    # ===== CUSTOMER =====
    "cs": "customer service",
    "csr": "customer service representative",
    "support": "support",

    # ===== GENERIC CLEAN =====
    "&": "and",
    "/": " ",
    "pt":'part time',
    'ft':'full time',
}

def clean_job_title(text, max_words_per_phrase=5):
    if not isinstance(text, str):
        return ""

    # --- normalize text: lowercase, dấu đặc biệt → space
    text = text.lower()
    text = re.sub(r"[/&\-]", " ", text)  # convert / & - → space
    text = re.sub(r"[^\w\s,]", " ", text)  # loại ký tự đặc biệt khác
    text = re.sub(r'\s+', ' ', text).strip()

    # --- split theo dấu ,
    phrases = [p.strip() for p in text.split(',') if p.strip()]

    cleaned_phrases = []
    for phrase in phrases:
        # --- apply synonym map trên phrase riêng lẻ
        for k, v in SYNONYM_MAP.items():
            phrase = re.sub(rf"\b{k}\b", v, phrase)

        # --- remove stopwords
        tokens = [w for w in phrase.split() if w not in STOP_WORDS]
        if not tokens:
            continue

        # --- lemmatize
        lemmatized = [lemmatizer.lemmatize(w) for w in tokens]

        # --- chỉ giữ phrases ngắn (tránh noise)
        if len(lemmatized) <= max_words_per_phrase:
            cleaned_phrases.append(' '.join(lemmatized))

    # --- dedup phrases giữ thứ tự
    dedup_phrases = list(dict.fromkeys(cleaned_phrases))

    return ', '.join(dedup_phrases).strip(', ')

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
import time
from multiprocessing import Pool, cpu_count

def process_skills(d):
    try:
        return clean_job_title(d)
    except:
        return []
st = time.time()

with Pool(cpu_count()) as p:
    results = p.map(process_skills, df3['job_title'])

job_title = results
df3['job_title'] = job_title

print(len(job_title), len(set(job_title)))

print(time.time() - st)

In [ ]:
st = time.time()

with Pool(cpu_count()) as p:
    results = p.map(process_skills, df3['job_skills'])

job_skills = results
df3['job_skills'] = job_skills
print(len(job_skills), len(set(job_skills)))

print(time.time() - st)

In [ ]:
df3['title_skills'] = df3['job_title'] + df3['job_skills']

In [ ]:
df3['title_skills'][0]

In [ ]:
skills = job_title + job_skills
t = []
for s in set(skills):
    t.extend(s.split())
len(set(t))

In [ ]:
df3.to_csv('1.3m Linkedin Jobs & Skills (2024) - clean.csv')

In [1]:
# import pandas as pd

# df = pd.read_csv('/kaggle/working/1.3m Linkedin Jobs & Skills (2024) - clean.csv').dropna()

In [ ]:
import time

st = time.time()
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=50000,   # tránh nổ RAM
    ngram_range=(1,2),
    min_df=100,              # bỏ từ hiếm
    max_df=0.8,            # bỏ từ quá phổ biến
    stop_words='english',
    token_pattern=r'(?u)\b[a-zA-Z]{2,}\b'
)

X = vectorizer.fit_transform(df3['title_skills'])  # sparse matrix
print(time.time() - st)

In [ ]:
vectorizer.get_feature_names_out()

In [ ]:
import numpy as np

job_titles = df3['job_title']
N = len(job_titles)          # tổng số job titles
min_idf_threshold = 5       # ngưỡng IDF
max_idf_threshold = 7
idf = vectorizer.idf_        # mảng IDF của tất cả feature
vocab = vectorizer.get_feature_names_out()

# chọn các từ hiếm (IDF > threshold)
rare_words_idx = np.where((idf > idf_threshold) & (idf < max_idf_threshold))[0]
rare_words = vocab[rare_words_idx]

mask = X[:, rare_words_idx].sum(axis=1).A1 > 0

df_filtered = df3[mask]
X_filtered = X[mask]
# tính số document chứa từ hiếm
df_counts = ((1 + N) / np.exp(idf[rare_words_idx] - 1)) - 1
df_counts = np.round(df_counts).astype(int)
print(min(df_counts))
for w, df in zip(rare_words, df_counts):
    print(w, df)
    break

len(df_filtered), len(df3['job_title'])

In [6]:
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = df["title_skills"].tolist()

pool = model.start_multi_process_pool(
    target_devices=["cuda:0", "cuda:1"]
)

chunk_size = 100000

# ❗ tránh giữ toàn bộ embeddings trong RAM
with open("embeddings.npy", "wb") as f:
    for i in tqdm(range(0, len(texts), chunk_size)):
        chunk = texts[i:i+chunk_size]

        emb = model.encode_multi_process(
            chunk,
            pool,
            batch_size=2048,
            normalize_embeddings=True  # 🔥 dùng luôn cho search
        )

        # lưu trực tiếp ra file (append)
        np.save(f, emb)

model.stop_multi_process_pool(pool)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  0%|          | 0/11 [00:00<?, ?it/s]/tmp/ipykernel_418/95915049.py:20: DeprecationWarning: The `encode_multi_process` method has been deprecated, and its functionality has been integrated into `encode`. You can now call `encode` with the same parameters to achieve multi-process encoding.
  emb = model.encode_multi_process(
100%|██████████| 11/11 [16:58<00:00, 92.63s/it]


In [17]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 76.9 MB/s eta 0:00:00:00:0100:01


In [18]:
import faiss

print(faiss.__version__)

1.13.2


In [19]:
import faiss
import numpy as np

d = 384  # dim của MiniLM
index = faiss.IndexFlatIP(d)  # cosine similarity

with open("/kaggle/working/embeddings.npy", "rb") as f:
    while True:
        try:
            emb = np.load(f)
            index.add(emb)  
        except (ValueError, EOFError):
            break

print("Total vectors:", index.ntotal)

Total vectors: 1057105


In [29]:
cv_text = clean_job_title("python machine learning data scientist")

query = model.encode([cv_text], normalize_embeddings=True)

D, I = index.search(query, k=10)

df.iloc[I[0]][["job_title", "job_skills"]]

,job_title,job_skills
575320,machine learning engineer,"machine learning, natural language processing,..."
1218599,senior python developer data engineering,"python, data engineering, sql, data science, s..."
455856,machine learning scientist,"python, sql, numpy, scipy, panda, scikitlearn,..."
300080,machine learning postdoctoral fellow,"machine learning, highenergy physic, artificia..."
44783,python data engineer,"python, data engineering, data pipeline, sql, ..."
1254780,senior data scientist uk,"machine learning, data analysis, python, numpy..."
836543,assistant professor digital medium,"artificial intelligence, machine learning, nat..."
283021,senior data scientist,"bioinformatics, applied mathematics, statistic..."
339688,machine learning scientist,"machine learning engineering, data analysis, s..."
641633,senior python developer data engineering,"python, data engineering, sql, data pipeline, ..."


In [30]:
def parse_skills(s):
    return set(s.lower().split(", "))

user_skills = set(["python", "sql"])

missing = []

for idx in I[0]:
    job_skills = parse_skills(df.iloc[idx]["job_skills"])
    missing += list(job_skills - user_skills)

from collections import Counter
print(Counter(missing).most_common(10))

[('statistic', 6), ('machine learning', 5), ('data science', 5), ('data analysis', 5), ('data engineering', 4), ('data pipeline', 4), ('pytorch', 3), ('scikitlearn', 3), ('numpy', 3), ('natural language processing', 2)]
